# Milestone 2

**Dataset Structure:**
Our dataset consists of 3-5 second audio clips recorded by our us using **https://dataforge-pk.vercel.app/**. We have organized them into the following directory structure 
* `raw/`
  * `english/` (angry, calm, excited, happy, sad, stressed)
  * `urdu/` (angry, calm, excited, happy, sad, stressed)

**Preprocessing Strategy:**
Since future milestones require us to implement custom Neural Networks and utilize Transfer Learning, we will extract **Mel Spectrograms** from our audio files. This effectively converts our audio classification problem into an image classification problem, allowing us to leverage pre-trained vision models like ResNet or VGG.

In [ ]:
!pip install imageio-ffmpeg
!pip install noisereduce librosa scikit-learn

In [11]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
from tqdm import tqdm # For progress bar
import warnings

import imageio_ffmpeg

warnings.filterwarnings('ignore') # Suppress librosa audioread warnings

BASE_PATH = os.path.expanduser("/DATASET/raw dataset/")

if not os.path.exists(BASE_PATH):
    print(f"Error: Path {BASE_PATH} does not exist. Please check your directory structure.")
else:
    print(f"Base path located at: {BASE_PATH}")

Error: Path /DATASET/raw dataset/ does not exist. Please check your directory structure.


## 1. Data Collection & Labeling Methodology
We will traverse our `raw/` directory. By reading the folder names we can automatically extract the `Language` and `Emotion` labels for each of our recordings file. We'll store this metadata in a Pandas DataFrame for an easy manipulation.

In [ ]:
data = []

# Iterating through the language folders

for language in os.listdir(BASE_PATH):
    lang_path = os.path.join(BASE_PATH, language)
    
    if os.path.isdir(lang_path):
        
        for emotion in os.listdir(lang_path): # Iterating through the emotion folders
            emo_path = os.path.join(lang_path, emotion)
            
            if os.path.isdir(emo_path):
                
                for file in os.listdir(emo_path): # Iterating through the .webm audio files
                    if file.endswith('.webm'):
                        file_path = os.path.join(emo_path, file)

                        data.append({
                            'filepath': file_path,
                            'language': language,
                            'emotion': emotion
                        })

# Creating the DataFrame

df = pd.DataFrame(data)
print(f"Total samples collected: {len(df)}")
display(df.head())

# DATASET STATISTICS

print("\n" + "="*40)
print("📊 DATASET STATISTICS (TABULAR FORM)")
print("="*40)

print("\n1. Total Emotion Counts:")
emotion_stats = df['emotion'].value_counts().reset_index()
emotion_stats.columns = ['Emotion', 'Total Count']
display(emotion_stats)


print("\n2. Total Language Counts:")
lang_stats = df['language'].value_counts().reset_index()
lang_stats.columns = ['Language', 'Total Count']
display(lang_stats)


print("\n3. Detailed Breakdown (Language vs. Emotion):")
lang_emo_table = pd.crosstab(index=df['language'], columns=df['emotion'], margins=True, margins_name="Total")
display(lang_emo_table)

## 2. Exploratory Data Analysis (EDA)

### 2.1. Distribution Bar
Checcking the distribution of our classes to see if our dataset is balanced. Imbalanced data can heavily skew our model later. We will also visualize the raw audio waveforms and their corresponding **Mel Spectrograms** to understand the features we are working with.

In [ ]:
plt.figure(figsize=(14, 5))

# Plot 1: Emotion Distribution
plt.subplot(1, 2, 1)
sns.countplot(data=df, x='emotion', palette='viridis', order=df['emotion'].value_counts().index)
plt.title('Distribution of Emotions')
plt.xlabel('Emotion')
plt.ylabel('Count')

# Plot 2: Language Distribution
plt.subplot(1, 2, 2)
sns.countplot(data=df, x='language', palette='Set2')
plt.title('Distribution of Languages')
plt.xlabel('Language')
plt.ylabel('Count')

plt.tight_layout()
plt.show()

### 2.2. Waveform and Spectrogram Visualization for all classes

In [ ]:
import os

try:
    import imageio_ffmpeg
    os.environ["PATH"] += os.pathsep + os.path.dirname(imageio_ffmpeg.get_ffmpeg_exe())

except ImportError:
    print("Warning: imageio_ffmpeg not found. Please run '!pip install imageio-ffmpeg' first.")

languages = df['language'].unique()
emotions = df['emotion'].unique()

print("Generating representative plots for each language and emotion...")

for lang in languages:
    print(f"\n{'-'*20} Language: {lang.upper()} {'-'*20}")
    
    fig, axes = plt.subplots(nrows=len(emotions), ncols=2, figsize=(16, 3 * len(emotions)), squeeze=False)
    
    for i, emo in enumerate(emotions):
        subset = df[(df['language'] == lang) & (df['emotion'] == emo)] 
        
        if subset.empty:

            print(f"Skipping {lang}-{emo}: No data found in DataFrame.")
            axes[i, 0].set_visible(False)
            axes[i, 1].set_visible(False)
            continue
            
        plot_successful = False
        
        for sample_file in subset['filepath']:
            try:
                y, sr = librosa.load(sample_file, sr=22050)
                
                # Plot Waveform

                ax_wave = axes[i, 0]
                librosa.display.waveshow(y, sr=sr, ax=ax_wave)
                ax_wave.set_title(f'{lang.capitalize()} - {emo.capitalize()}: Waveform')
                ax_wave.set_xlabel('Time (s)')
                ax_wave.set_ylabel('Amplitude')
                
                # Plot Mel Spectrogram

                ax_spec = axes[i, 1]
                mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128)
                mel_spect_db = librosa.power_to_db(mel_spect, ref=np.max)
                
                img = librosa.display.specshow(mel_spect_db, sr=sr, x_axis='time', y_axis='mel', ax=ax_spec)
                ax_spec.set_title(f'{lang.capitalize()} - {emo.capitalize()}: Mel Spectrogram')
                fig.colorbar(img, ax=ax_spec, format='%+2.0f dB')
                
                plot_successful = True
                break
                
            except Exception as e:

                print(f"  -> Skipping broken file: {sample_file} ({e})")
                continue
                
        if not plot_successful:
            print(f"FAILED to process ANY files for {lang}-{emo}. All files appear corrupted.")
            axes[i, 0].set_visible(False) 
            axes[i, 1].set_visible(False)
            
    plt.tight_layout()
    plt.show()

## 3. Pre-processing Pipeline
Neural networks require the fixed-size inputs. Since our recordings will vary between 3 to 5 seconds therefore  we need to standardize their lengths. 

**Steps:**
1. **Standardize Length:** Pad audio shorter than 4 seconds with zeros, and truncate audio longer than 4 seconds.
2. **Feature Extraction:** Extract the Mel Spectrogram from the standardized audio.
3. **Encoding Labels:** Convert categorical emotion labels into numerical values.


In [ ]:
import noisereduce as nr
from sklearn.preprocessing import LabelEncoder

TARGET_SR = 22050
DURATION = 4 # Forcing all audio to exactly 4 seconds
SAMPLES = TARGET_SR * DURATION 

def extract_features(y, sr):
    # Extract Mel Spectrogram
    mel_spect = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=128, fmax=8000)
    mel_spect_db = librosa.power_to_db(mel_spect, ref=np.max)
    
    # NORMALIZATION: Standardize the spectrogram (Mean = 0, Std = 1)
    mean = np.mean(mel_spect_db)
    std = np.std(mel_spect_db)
    if std != 0:
        mel_spect_db = (mel_spect_db - mean) / std
        
    return mel_spect_db

def add_white_noise(y):
    noise_amplitude = 0.005 * np.random.uniform() * np.amax(y)
    y_noise = y + noise_amplitude * np.random.normal(size=y.shape[0])
    return y_noise

print("Starting preprocessing pipeline with Cleaning, Trimming, Augmentation and Normalization...")

processed_data = []

for index, row in tqdm(df.iterrows(), total=df.shape[0]): 
    filepath = row['filepath']
    emotion = row['emotion']
    language = row['language']
    
    try:
        y, sr = librosa.load(filepath, sr=TARGET_SR) 
        
        # 1. TRIM SILENCE: Remove leading/trailing silence (top_db=20 is a standard threshold)
        y_trimmed, _ = librosa.effects.trim(y, top_db=20)
        
        # 2. FILTER BACKGROUND NOISE: Clean up the trimmed audio
        y_denoised = nr.reduce_noise(y=y_trimmed, sr=sr)
        
        # 3. STANDARDIZE LENGTH: Now truncate or pad the clean, trimmed audio
        if len(y_denoised) > SAMPLES: 
            y_final = y_denoised[:SAMPLES] # Truncate
        else:
            padding = SAMPLES - len(y_denoised)
            y_final = np.pad(y_denoised, (0, padding), 'constant') # Pad with zeros
            
        # Extract features from the finalized clean audio
        original_features = extract_features(y_final, TARGET_SR)
        processed_data.append({
            'features': original_features,
            'language': language,
            'emotion': emotion,
            'is_augmented': False
        })
        
        # AUGMENTATION: Create a noisy version of the cleaned file to double dataset size
        y_augmented = add_white_noise(y_final)
        augmented_features = extract_features(y_augmented, TARGET_SR)
        processed_data.append({
            'features': augmented_features,
            'language': language,
            'emotion': emotion,
            'is_augmented': True
        })
        
    except Exception as e:
        print(f"Error processing {filepath}: {e}")

processed_df = pd.DataFrame(processed_data)
print(f"\nPreprocessing complete! Total samples (including augmented): {len(processed_df)}")

In [ ]:
print("\nEncoding Labels...")

# ENCODING: Converting string labels to integers for Neural Networks

emotion_encoder = LabelEncoder()
language_encoder = LabelEncoder()

processed_df['emotion_encoded'] = emotion_encoder.fit_transform(processed_df['emotion'])
processed_df['language_encoded'] = language_encoder.fit_transform(processed_df['language'])

print("Emotion Classes Mapping:", dict(zip(emotion_encoder.classes_, emotion_encoder.transform(emotion_encoder.classes_))))
print("Language Classes Mapping:", dict(zip(language_encoder.classes_, language_encoder.transform(language_encoder.classes_))))

# Separate features (X) and labels (y)

X = np.array(processed_df['features'].tolist())
y_emotion = np.array(processed_df['emotion_encoded'].tolist())
y_language = np.array(processed_df['language_encoded'].tolist())

# Note: Removed the redundant librosa.power_to_db here since it was already done in Cell 12!

X = X[..., np.newaxis] 

# Normalization (Min-Max Scaling to [0, 1] per spectrogram)

X_min = X.min(axis=(1, 2, 3), keepdims=True)
X_max = X.max(axis=(1, 2, 3), keepdims=True)
X_normalized = (X - X_min) / (X_max - X_min + 1e-8) # Added a tiny epsilon (1e-8) to prevent division by zero errors

X_ready = np.repeat(X_normalized, 3, axis=-1)


processed_dir = os.path.expanduser("~/Desktop/Machine Learning CEP/processed/")
os.makedirs(processed_dir, exist_ok=True)

np.save(os.path.join(processed_dir, 'X_features.npy'), X_ready) 
np.save(os.path.join(processed_dir, 'y_emotion.npy'), y_emotion)
np.save(os.path.join(processed_dir, 'y_language.npy'), y_language)

print(f"\nPreprocessed data successfully saved to {processed_dir}")
print(f"Final X shape ready for ResNet/VGG: {X_ready.shape}")

In [ ]:
# AUGMENTED DATASET STATISTICS

print("\n" + "="*40)
print("📊 AUGMENTED DATASET STATISTICS (POST-PROCESSING)")
print("="*40)

# Total Emotion Counts after Augmentation

aug_emotion_stats = processed_df['emotion'].value_counts().reset_index()
aug_emotion_stats.columns = ['Emotion', 'Total Count (Original + Augmented)']
display(aug_emotion_stats)

# Detailed Breakdown (Language vs. Emotion) after Augmentation

aug_lang_emo_table = pd.crosstab(index=processed_df['language'], columns=processed_df['emotion'], margins=True, margins_name="Total")
display(aug_lang_emo_table)